# Business Analytics Project: Synthetic Project Data Analysis

This notebook presents a complete workflow for exploring and modeling a synthetic dataset related to project management. The dataset includes various features such as team size, project duration, budget, complexity, and risk level. We perform exploratory data analysis to understand the distribution of variables and relationships between them, followed by predictive modeling to estimate project outcomes.

This project is intended for Business Analysts, Program Managers, and Data Analysts who want to showcase their ability to work with data, generate insights, and build machine learning models. The dataset is synthetic but designed to mimic real-world project characteristics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, mean_squared_error

# Configure plots
sns.set(style='whitegrid')
%matplotlib inline


In [ ]:
# Load the synthetic project dataset
file_path = 'synthetic_project_data.csv'
df = pd.read_csv(file_path)
print(f'Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.')
df.head()


## Summary Statistics

We start by examining basic statistics and data types to get a sense of the dataset's structure.


In [ ]:
df.describe(include='all')


## Exploratory Data Analysis and Visualizations

Let's explore the distribution of some key variables and relationships between them.
- **Project Duration**: Histogram of project durations.
- **Risk Level Counts**: Bar chart showing the number of projects at each risk level.
- **Team Size vs Customer Satisfaction**: Scatter plot to see how team size influences customer satisfaction.


In [ ]:
# Histogram of project durations
plt.figure(figsize=(8, 4))
sns.histplot(df['project_duration_days'], bins=20, kde=True)
plt.title('Distribution of Project Duration (Days)')
plt.xlabel('Duration (days)')
plt.ylabel('Frequency')
plt.show()

# Bar chart of risk level counts
plt.figure(figsize=(6, 4))
sns.countplot(x='risk_level', data=df, order=sorted(df['risk_level'].unique()))
plt.title('Number of Projects by Risk Level')
plt.xlabel('Risk Level')
plt.ylabel('Count')
plt.show()

# Scatter plot: team size vs customer satisfaction
plt.figure(figsize=(8, 5))
sns.scatterplot(x='team_size', y='customer_satisfaction', hue='risk_level', data=df)
plt.title('Team Size vs Customer Satisfaction by Risk Level')
plt.xlabel('Team Size')
plt.ylabel('Customer Satisfaction')
plt.show()


## Correlation Analysis

Understanding the correlation between numeric variables helps identify potential predictors for modeling.


In [ ]:
# Compute correlation matrix for numeric variables
numeric_cols = ['team_size', 'project_duration_days', 'budget_k', 'complexity', 'customer_satisfaction']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

corr_matrix


## Predictive Modeling: On-Time Completion (Classification)

We build a logistic regression model to predict whether a project will be completed on time (\`on_time\`). We use features such as team size, project duration, budget, complexity, and risk level. Categorical variables are encoded via one-hot encoding.


In [ ]:
# Features and target for classification
X_clf = df[['team_size', 'project_duration_days', 'budget_k', 'complexity', 'risk_level']]
y_clf = df['on_time']

# Preprocess: one-hot encode the categorical column 'risk_level'
categorical_features = ['risk_level']
numeric_features = ['team_size', 'project_duration_days', 'budget_k', 'complexity']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ])

# Create pipeline with logistic regression
clf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=200))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# Train the model
clf_model.fit(X_train, y_train)

# Evaluate
y_pred = clf_model.predict(X_test)
print('Classification Report (On-Time Completion):')
print(classification_report(y_test, y_pred))


## Predictive Modeling: Customer Satisfaction (Regression)

Next we predict customer satisfaction scores based on other features using linear regression. We'll one-hot encode the risk level and evaluate the model using root mean squared error (RMSE).


In [ ]:
# Features and target for regression
X_reg = df[['team_size', 'project_duration_days', 'budget_k', 'complexity', 'risk_level']]
y_reg = df['customer_satisfaction']

# Preprocess: one-hot encode 'risk_level'
preprocessor_reg = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), ['risk_level']),
        ('num', 'passthrough', ['team_size', 'project_duration_days', 'budget_k', 'complexity'])
    ])

# Build pipeline for linear regression
reg_model = Pipeline(steps=[
    ('preprocessor', preprocessor_reg),
    ('regressor', LinearRegression())
])

# Train-test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Train the model
reg_model.fit(X_train_reg, y_train_reg)

# Predictions and evaluation
y_pred_reg = reg_model.predict(X_test_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg, squared=False)
print(f'Root Mean Squared Error (Customer Satisfaction): {rmse:.2f}')


## Conclusions

This notebook demonstrated a full end-to-end workflow for exploring and modeling a synthetic project management dataset. Key takeaways:

- **Exploratory Data Analysis** revealed the distribution of project durations, the balance of risk levels, and relationships between team size and customer satisfaction.
- **Correlation Analysis** highlighted correlations among numeric features, which informed the selection of predictors for modeling.
- **Predictive Modeling** results:
  - Logistic regression provided insights into factors influencing on-time completion, showing how variables such as project duration and risk level impact outcomes.
  - Linear regression predicted customer satisfaction scores with reasonable accuracy, indicating which features contribute most to satisfaction.

Feel free to extend this notebook by experimenting with additional models, tuning hyperparameters, or creating dashboards to present insights.
